# 04 · From the frame-window baseline to the clip model

This notebook records a negative result honestly, then the model that replaced it.

**What the obvious approach gave.** 291 window statistics, 22,662 overlapping
2 s windows, RandomForest(depth 12), participant-level split:

| Exercise | train AUC | test AUC (unseen people) |
|---|---|---|
| squat | 0.998 | 0.722 |
| bicep_curl | 0.998 | **0.544** |
| push_up | 1.000 | 0.749 |

AUC 0.998 in-sample means the model memorised the participants. 22k nearly
identical windows from 17 people cannot teach generalisation.

Two further findings from the diagnostic experiments:

* Removing ``body_scale`` and the visibility columns (which encode *where and how
  far the camera was*, i.e. recording identity) does not fix it.
* The labelled BAD clips are almost geometrically identical to the GOOD ones:
  mean elbow ROM of a GOOD curl = 120.1 deg, of a BAD curl = 120.0 deg. The
  difference the annotators saw is cadence/control, not joint angles.

So the final model scores **one clip per sample** with pose + repetition-tempo
features, and is validated with 5-fold GroupKFold on participants.

## Shared setup (run first)

This notebook uses the modules in `ml/`. Upload them once to the Colab runtime:

1. Click the folder icon in the left sidebar (Files).
2. Click **Upload to session storage** and select `extract_pose.py`,
   `fitness_features.py`, `build_dataset.py`, `train_baseline.py`,
   `train_temporal.py`, `export_onnx.py` from `F:\Fitness Coach\ml`.
3. Run the cell below.

The dataset zip is expected at
`/content/drive/MyDrive/Dataset Exercise Quality-wise.zip` (Google Drive).

In [ ]:
import os, sys, subprocess, json, glob
from pathlib import Path

ROOT = Path('/content/fitness')
ROOT.mkdir(parents=True, exist_ok=True)

# The uploaded modules land in /content; mirror them into the work root.
for f in ['extract_pose.py', 'fitness_features.py', 'build_dataset.py', 'train_baseline.py',
          'experiment_generalization.py', 'experiment_clip.py', 'experiment_repmetry.py',
          'train_final.py', 'train_temporal.py', 'export_onnx.py']:
    src = Path('/content') / f
    if src.exists():
        dst = ROOT / f
        if not dst.exists() or src.stat().st_mtime > dst.stat().st_mtime:
            dst.write_bytes(src.read_bytes())

sys.path.insert(0, str(ROOT))
print('work root :', ROOT)
print('modules   :', sorted(p.name for p in ROOT.glob('*.py')))
print('cuda      :', subprocess.run('nvidia-smi -L', shell=True, capture_output=True, text=True).stdout.strip() or 'no GPU')

## 1. Keep the record of the window baseline

In [ ]:
!cd /content/fitness && python -u /content/fitness/train_baseline.py

## 2. Diagnostic A - do session-identity columns explain the overfit?

5-fold GroupKFold, several feature sets and models, per exercise.

In [ ]:
# ~10-20 min on the free 2 vCPU runtime; safe to skip, the result is recorded in
# dataset/generalization_experiment.csv
!cd /content/fitness && python -u /content/fitness/experiment_generalization.py

## 3. Diagnostic B - clip-level domain features

One sample per clip from quantities a coach names out loud: ROM, torso lean,
knee tracking, depth, sway, symmetry. Fast (444 rows).

In [ ]:
!cd /content/fitness && python -u /content/fitness/experiment_clip.py

## 4. Diagnostic C - is the label a tempo difference?

Percentile-threshold hysteresis rep detector -> cadence, rhythm consistency,
time spent near the bottom, ROM per rep. Also prints a **univariate AUC table**,
which is the fastest way to see whether a signal exists at all.

In [ ]:
!cd /content/fitness && python -u /content/fitness/experiment_repmetry.py

## 5. Train the final model (pose + tempo, one clip = one sample)

Fits on train participants, picks the decision threshold on validation
participants (max F1 subject to a 20% false-accept ceiling), reports held-out
test participants, then refits on train+val and exports ONNX with a
sklearn-vs-onnxruntime parity check.

In [ ]:
!cd /content/fitness && python -u /content/fitness/train_final.py

## 2. Audit the split (who is in which set)

In [ ]:
import json
splits = json.load(open(ROOT / 'dataset/splits.json'))
for k, v in splits.items():
    print(f'{k:5s} ({len(v):2d} participants): {v}')
common = set(splits['train']) & set(splits['test'])
print('\noverlap train/test (must be empty):', common)

## 6. Results table

In [ ]:
import json, pandas as pd
fm = json.load(open(ROOT / 'export/final_metrics.json'))
rows = []
for ex, r in fm['exercises'].items():
    rows.append({'exercise': ex, 'clips': r['n_clips'],
                 'cv_auc': r['cv_5fold_group']['roc_auc'],
                 'cv_f1': r['cv_5fold_group']['f1'],
                 'test_auc': r['test']['roc_auc'], 'test_f1': r['test']['f1'],
                 'test_acc': r['test']['accuracy'],
                 'false_accept': r['test']['false_accept_rate_bad_as_good'],
                 'threshold': r['threshold']})
print(pd.DataFrame(rows).to_string(index=False))
print()
for ex, r in fm['exercises'].items():
    print(ex, 'top features:', list(r['top_features'])[:6])

## 8. How to read this

* `cv_auc` (5-fold GroupKFold over participants) is the most stable number: it
  uses every clip and never tests on a training participant.
* `test_auc` / `test_f1` use only 4 held-out participants (~24 clips), so a
  0.05-0.15 swing between reruns is noise, not progress.
* If `cv_auc` is near 0.5 the model has learned nothing generalisable - that is
  the case for bicep curls with this dataset, whatever the in-sample numbers say.
* `false_accept` (BAD form scored GOOD) is kept low by the threshold rule, which
  is why precision/recall are not balanced.

## 7. Per-camera-view breakdown

In [ ]:
# The final model is clip-level, so per-view breakdowns are computed directly on
# dataset/clip_features.parquet + dataset/rep_features.parquet. Quick check of
# whether a view is systematically easier:
import pandas as pd, numpy as np, joblib
clip = pd.read_parquet(ROOT / 'dataset/clip_features.parquet')
for ex in ['squat', 'bicep_curl', 'push_up']:
    b = joblib.load(ROOT / f'models/final_{ex}.joblib')
    d = clip[clip.exercise == ex]
    X = d[b['feature_cols']].to_numpy(np.float32)
    d = d.assign(p=b['pipeline'].predict_proba(X)[:, 1],
                 pred=(b['pipeline'].predict_proba(X)[:, 1] >= b['threshold']).astype(int))
    print(f'--- {ex} ---')
    print(d.groupby('view').apply(lambda g: pd.Series({
        'n': len(g), 'accuracy': round((g.pred == g.label).mean(), 3),
        'mean_p_good': round(g.p.mean(), 3)}), include_groups=False).to_string())